[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takeshun1984/NumeralAnalysisInGeophysics_SolidEarth/blob/main/09_FDM_2nd_layered_P-SV_CerjanZB_numba.ipynb)

### Numbaによる並列化版

`09_FDM_2nd_layered_P-SV_CerjanZB.ipynb`と同じ計算を、Numbaで並列化して高速化したもの。Numbaによる実装の解説は`08_FDM_2nd_homo_P-SV_CerjanZB_numba.ipynb`を参照。結果は`output/09numba`に出力するので、09の結果（`output/09`）と比較できる。


In [ ]:
import numpy as np
import os

# 領域設定など
SP = np.float32
NX, NZ = 800, 250
KFS = 25
DX, DZ = 0.4, 0.4
DT = 0.02
NTMAX = 6000

EPS = 1.0e-15

# 震源情報
X0 = 40.0
Z0 = 15.0
I0 = int(X0/DX)
K0 = int(Z0/DZ)+KFS
# Z0/DZ, X0/DX が割り切れない場合は切り捨てられるので、実際の震源位置を表示
print(f"震源位置: x = {I0*DX:.2f} km, z = {(K0-KFS)*DZ:.2f} km")
T0, TS = 3.0, 0.0
MXX, MZZ, MXZ, MO = -1.0, 1.0, 0.0, 1.0
DTXZ = DT / (DX * DZ)

# --- 出力・減衰設定 ---
NTDEC, NXD, NZD = 100, 2, 2
DST = 10.0
NST = int(NX * DX / DST) - 1
NSKIP = 2
NWMAX = NTMAX // NSKIP
OUTDIR = "output/09numba"  # 09（NumPy版）と結果を比較できるよう出力先を分ける
ONAME0, WNAME0 = f"{OUTDIR}/psv.l.", f"{OUTDIR}/wav.l."

F0 = 1/T0

# --- 関数定義 ---
def kupper(t, ts, tr):
    if ts <= t <= ts + tr:
        return 3 * np.pi * (np.sin(np.pi * (t - ts) / tr))**3 / (4 * tr)
    else:
        return 0.0

# 配列の初期化 0~NX+1
SXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
SZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
SXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
VX  = np.zeros((NX + 2, NZ + 2), dtype=SP)
VZ  = np.zeros((NX + 2, NZ + 2), dtype=SP)

RXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
RZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
RXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)

# 波形記録用配列
VWX = np.zeros((NWMAX, NST), dtype=SP)
VWZ = np.zeros((NWMAX, NST), dtype=SP)
ISTX = np.array([int((ist+1)*DST/DX) for ist in range(NST)])
ISTZ = KFS+1

# --- 吸収境界条件 (Cerjan 1985) パラメータ ---
BETA, NA = 0.09, 20
gx1, gx2 = np.ones(NX+2, dtype=SP), np.ones(NX+2, dtype=SP)
gz1, gz2 = np.ones(NZ+2, dtype=SP), np.ones(NZ+2, dtype=SP)
for i in range(1, NA + 1):
    val1 = np.exp(-BETA*(1.0-(i-0.5)/NA)**2)
    val2 = np.exp(-BETA*(1.0-i/NA)**2)
    
    gx1[i], gz1[i], gx2[i], gz2[i] = val1, val1, val2, val2

    gx1[NX-i+1], gx2[NX-i+1], gz1[NZ-i+1], gz2[NZ-i+1] = val2, val1, val2, val1

GX1, GZ1 = np.meshgrid(gx1, gz1, indexing='ij')
GX2, GZ2 = np.meshgrid(gx2, gz2, indexing='ij')

In [ ]:

# 均質媒質
VP, VS, RO = 6.0, 3.5, 2.3
RIG_val = RO * VS**2
LAM_val = RO * VP**2 - 2.0 * RO * VS**2

# 各格子点に物性を割り当て
RIG = np.full((NX + 2, NZ + 2), RIG_val, dtype=SP)
LAM = np.full((NX + 2, NZ + 2), LAM_val, dtype=SP)
RHO = np.full((NX + 2, NZ + 2), RO,      dtype=SP)
QP  = np.full((NX + 2, NZ + 2), 600.,    dtype=SP)
QS  = np.full((NX + 2, NZ + 2), 300.,    dtype=SP)

for k in range(NZ+2):
    z = (k-KFS)*DZ
    if z <= 0.0  : 
        vp, vs, ro, Q0 = 0.00, 0.00, 0.001, 10.0
    elif z < 3.0 : 
        vp, vs, ro, Q0 = 5.50, 3.14, 2.3, 300
    elif z < 18.0: 
        vp, vs, ro, Q0 = 6.00, 3.55, 2.4, 300
    elif z < 33.0: 
        vp, vs, ro, Q0 = 6.70, 3.83, 2.8, 300
    else: 
        vp, vs, ro, Q0 = 7.80, 4.46, 3.2, 300

    RIG[:, k], RHO[:, k] = ro * vs**2, ro
    LAM[:, k] = ro * vp**2 - 2.0 * ro * vs**2
    QP [:, k], QS [:, k] = 2*Q0, Q0

ここで、2次精度の場合、波長あたり15格子以上必要なため、
$$
f^{max}=\frac{V_S^{min}}{15\Delta x} = \frac{3.14}{15\times 0.4} \sim 0.52
$$
と0.52 Hzまで精度よく計算できることが予想される。

In [ ]:
W0   = 2.0 * np.pi * F0
TAU  = 1.0 / W0 * (np.sqrt(1.0 + QP**(-2)) - 1.0 / QP)
TAUP_ratio = 1.0 / W0**2 / TAU**2
TAUS_ratio = (1.0 + W0 * TAU * QS) / (W0 * QS * TAU - (W0 * TAU)**2)
TU   = -1.0 / TAU

以下、不均質媒質（特に固体気体境界）を導入したため、以下のような平均媒質を計算する。

In [ ]:
rigxz = np.full((NX + 2, NZ + 2), 0.00, dtype=SP)
bx    = np.full((NX + 2, NZ + 2), 0.00, dtype=SP)
bz    = np.full((NX + 2, NZ + 2), 0.00, dtype=SP)

bx[1:NX+1,1:NZ+1] = 2.0/(RHO[1:NX+1,1:NZ+1]+RHO[2:NX+2,1:NZ+1])
bz[1:NX+1,1:NZ+1] = 2.0/(RHO[1:NX+1,1:NZ+1]+RHO[1:NX+1,2:NZ+2])

rig00 = RIG[1:NX+1,1:NZ+1]
rig10 = RIG[2:NX+2,1:NZ+1]
rig01 = RIG[1:NX+1,2:NZ+2]
rig11 = RIG[2:NX+2,2:NZ+2]
rigxz = 4*rig00*rig01*rig10*rig11 /                \
                     ( rig00*rig01*rig10 + rig00*rig01*rig11 +     \
                       rig00*rig10*rig11 + rig01*rig10*rig11 + EPS )

### Numbaによる更新関数

`08_FDM_2nd_homo_P-SV_CerjanZB_numba.ipynb`と同じ関数で、不均質媒質のため以下の2点だけが異なる（平均化の考え方は`09_FDM_2nd_layered_P-SV_CerjanZB.ipynb`を参照）。

- $\sigma_{xz}$の剛性率に、調和平均`rigxz`（関数内では`RIGXZ`）を用いる。
- 速度の更新に、$1/\rho$の代わりに密度の平均の逆数`bx`、`bz`を用いる。


In [ ]:
import numba
from numba import njit, prange

print(f"Numba {numba.__version__}, 使用スレッド数: {numba.get_num_threads()}")

# SXZ の位置の剛性（調和平均 rigxz）を、他の配列と同じ添字 [i, k] で参照できるよう全体サイズの配列に
RIGXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
RIGXZ[1:NX+1, 1:NZ+1] = rigxz

@njit(parallel=True, fastmath=True)
def update_stress(SXX, SZZ, SXZ, RXX, RZZ, RXZ, VX, VZ,
                  LAM, RIG, RIGXZ, TU, TAUP_ratio, TAUS_ratio,
                  GX1, GZ1, GX2, GZ2, NX, NZ, rdx, rdz, dt):
    """メモリ変数と応力場の更新（Cerjanの吸収境界を含む）"""
    for i in prange(1, NX+1):          # i 方向を複数スレッドで分担
        for k in range(1, NZ+1):       # k 方向（メモリ上で連続）を内側に
            # 速度の空間微分
            dxvx = (VX[i  , k  ] - VX[i-1, k  ]) * rdx
            dzvz = (VZ[i  , k  ] - VZ[i  , k-1]) * rdz
            dxvz = (VZ[i+1, k  ] - VZ[i  , k  ]) * rdx
            dzvx = (VX[i  , k+1] - VX[i  , k  ]) * rdz

            lam, rig, rigxz = LAM[i, k], RIG[i, k], RIGXZ[i, k]
            tu, taup, taus = TU[i, k], TAUP_ratio[i, k], TAUS_ratio[i, k]
            den = 1.0 - tu*dt*0.5

            # メモリ変数更新 (Crank-Nicolson)
            rxxn, rzzn, rxzn = RXX[i, k], RZZ[i, k], RXZ[i, k]
            RXX[i, k] = (rxxn + tu*(rxxn*0.5 + (lam+2.0*rig)*(taup-1.0)*(dxvx+dzvz) - 2.0*rig*(taus-1.0)*dzvz)*dt) / den
            RZZ[i, k] = (rzzn + tu*(rzzn*0.5 + (lam+2.0*rig)*(taup-1.0)*(dxvx+dzvz) - 2.0*rig*(taus-1.0)*dxvx)*dt) / den
            RXZ[i, k] = (rxzn + tu*(rxzn*0.5 + rigxz*(taus-1.0)*(dxvz+dzvx))*dt) / den

            # 応力場更新 + 吸収境界
            SXX[i, k] = (SXX[i, k] + ((lam+2.0*rig)*taup*(dxvx+dzvz) - 2.0*rig*taus*dzvz + (RXX[i, k]+rxxn)*0.5)*dt) * GX1[i, k]*GZ1[i, k]
            SZZ[i, k] = (SZZ[i, k] + ((lam+2.0*rig)*taup*(dxvx+dzvz) - 2.0*rig*taus*dxvx + (RZZ[i, k]+rzzn)*0.5)*dt) * GX1[i, k]*GZ1[i, k]
            SXZ[i, k] = (SXZ[i, k] + (rigxz*taus*(dxvz+dzvx) + (RXZ[i, k]+rxzn)*0.5)*dt) * GX2[i, k]*GZ2[i, k]


@njit(parallel=True, fastmath=True)
def update_velocity(VX, VZ, SXX, SZZ, SXZ, bx, bz,
                    GX1, GZ1, GX2, GZ2, NX, NZ, rdx, rdz, dt):
    """速度場の更新（Cerjanの吸収境界を含む）"""
    for i in prange(1, NX+1):
        for k in range(1, NZ+1):
            # 応力の空間微分
            dxsxx = (SXX[i+1, k  ] - SXX[i  , k  ]) * rdx
            dxsxz = (SXZ[i  , k  ] - SXZ[i-1, k  ]) * rdx
            dzszz = (SZZ[i  , k+1] - SZZ[i  , k  ]) * rdz
            dzsxz = (SXZ[i  , k  ] - SXZ[i  , k-1]) * rdz

            VX[i, k] = (VX[i, k] + (dxsxx + dzsxz) * bx[i, k] * dt) * GX2[i, k]*GZ1[i, k]
            VZ[i, k] = (VZ[i, k] + (dxsxz + dzszz) * bz[i, k] * dt) * GX1[i, k]*GZ2[i, k]


In [ ]:
# メインループ
import time

rDX, rDZ, DT_SP = SP(1.0/DX), SP(1.0/DZ), SP(DT)   # 単精度の定数として関数に渡す

print(f"{'Step':>5} / {NTMAX}: {'Time':>7} {'Vxmax':>12}")
os.makedirs(OUTDIR, exist_ok=True)

# 再実行しても前回の結果の続きにならないよう、波動場を初期化
for arr in (SXX, SZZ, SXZ, VX, VZ, RXX, RZZ, RXZ, VWX, VWZ):
    arr[:] = 0.0

t_start = time.perf_counter()
for it in range(1, NTMAX + 1):
    T = it * DT

    # 1. メモリ変数・応力場の更新
    update_stress(SXX, SZZ, SXZ, RXX, RZZ, RXZ, VX, VZ,
                  LAM, RIG, RIGXZ, TU, TAUP_ratio, TAUS_ratio,
                  GX1, GZ1, GX2, GZ2, NX, NZ, rDX, rDZ, DT_SP)

    # 震源注入
    sdrop = MO * kupper(T, TS, T0) * DTXZ
    SXX[I0, K0] -= MXX * sdrop
    SZZ[I0, K0] -= MZZ * sdrop
    SXZ[I0-1:I0+1, K0-1:K0+1] -= MXZ * sdrop * 0.25

    # 2. 速度場の更新
    update_velocity(VX, VZ, SXX, SZZ, SXZ, bx, bz,
                    GX1, GZ1, GX2, GZ2, NX, NZ, rDX, rDZ, DT_SP)

    # 波形記録
    if it % NSKIP == 0:
        it1 = (it // NSKIP) - 1
        if it1 < NWMAX:
            VWX[it1, :] =  VX[ISTX, ISTZ]
            VWZ[it1, :] = -VZ[ISTX, ISTZ] # Convert “up” to “positive”

    # 進捗表示とスナップショット出力
    if it % NTDEC == 0:
        vxmax = np.abs(VX).max()
        print(f"{it:5d}/{NTMAX:5d}: T={T:6.2f}[s] vxmax={vxmax:.3e}")
        
        # 空間データ一括計算
        i_idx, k_idx = np.arange(1, NX+1, NXD), np.arange(1, NZ+1, NZD)
        ii, kk = np.meshgrid(i_idx, k_idx, indexing='ij')
        
        # 物理量の計算 (div, rot)
        d_vx_x = (VX[ii  ,kk  ] - VX[ii-1, kk  ]) / DX
        d_vz_z = (VZ[ii  ,kk  ] - VZ[ii  , kk-1]) / DZ
        d_vx_z = (VX[ii  ,kk+1] - VX[ii  , kk  ]) / DZ
        d_vz_x = (VZ[ii+1,kk  ] - VZ[ii  , kk  ]) / DX
        
        out_data = np.column_stack([
            (ii*DX).ravel(), ((kk-KFS)*DZ).ravel(),
            VX[ii, kk].ravel(), VZ[ii, kk].ravel(),
            (d_vx_x + d_vz_z).ravel(), (d_vz_x - d_vx_z).ravel()
        ])
        np.savetxt(f"{ONAME0}{it:05d}.out", out_data, fmt='%9.3f %9.3f %12.3e %12.3e %12.3e %12.3e')

print(f"計算時間: {time.perf_counter() - t_start:.1f} s（初回はコンパイル時間を含む）")

# --- 波形データの最終保存 ---
print("波形データ保存中...")
time_axis = np.arange(1, NWMAX + 1) * DT * NSKIP
for ist in range(NST):
    wfname = f"{WNAME0}{int(ISTX[ist]*DX):04d}.dat"
    np.savetxt(wfname, np.column_stack([time_axis, VWX[:, ist], VWZ[:, ist]]), fmt='%9.3f %14.5e %14.5e')

print("完了")

波動場の描画の際に、1次元構造の境界面深さを破線で示すようにした。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor

time_val = 30.0
step = int(round(time_val/DT/NTDEC))*NTDEC # 出力間隔NTDECの倍数に丸める
time_val = step*DT
filename = f"{ONAME0}{step:05d}.out"

fig, axs = plt.subplots(2, 2, figsize=(15, 8))
plt.rcParams.update({'font.size': 10})

# データ整形
out = pd.read_csv(filename, sep=r'\s+', names=('X','Z','VX','VZ','div','rot'))

nxs = len(np.unique(out.X))
nzs = len(np.unique(out.Z))

X  = out["X"].values.reshape(nxs, nzs)
Z  = out["Z"].values.reshape(nxs, nzs)
vx_s = out["VX"].values.reshape(nxs, nzs)
vz_s = out["VZ"].values.reshape(nxs, nzs)
div_s = out["div"].values.reshape(nxs, nzs)
rot_s = out["rot"].values.reshape(nxs, nzs)
    
# 正規化
vx_s *= fact
vz_s *= fact
div_s *= fact*5
rot_s *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, vx_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, vz_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, div_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}} \times 5$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X from source [km]',fontsize=14)
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, rot_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X from source [km]',fontsize=14)

xb = np.arange(0,321,1)
for ax in axs.flatten():
    ax.set_xlim(0 , 320)
    ax.set_ylim(90, -10)
    ax.set_aspect(1.0)
    ax.plot(xb,np.full_like(xb,0),ls='-',color='black')
    ax.plot(xb,np.full_like(xb,3),ls='--',color='black')
    ax.plot(xb,np.full_like(xb,18),ls='--',color='black')
    ax.plot(xb,np.full_like(xb,33),ls='--',color='black')
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()

以下、出力した波形データを利用して距離と地震波形の時系列の関係を図示している。

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import glob
import os

# 1. 設定
folder_path = OUTDIR  # ファイルがあるディレクトリ
file_pattern = 'wav.l.*.dat'
files = sorted(glob.glob(os.path.join(folder_path, file_pattern)))
fact = 3e4

sampling_rate = 1/(DT*NSKIP)
F1 = 0.05
F2 = 0.52
order = 2

nyquist = 0.5 * sampling_rate
low  = F1 / nyquist
high = F2 / nyquist

plt.figure(figsize=(6, 4))

# 2. データの読み込みとプロット
for file in files:
    # ファイル名から距離(km)を抽出 (例: wav.h.0010.dat -> 10)
    filename = os.path.basename(file)
    distance_km = int(filename.split('.')[2]) 
    
    try:
        data = np.loadtxt(file)
        time = data[:, 0]
        vz = data[:, 2]*fact
        vx = data[:, 1]*fact
        # 0 : time
        # 1 : Vx
        # 2 : Vz
        sos = signal.butter(order, [low, high], btype='band', output='sos')
        vz_filt = signal.sosfiltfilt(sos, vz)
        #vx_filt = signal.sosfiltfilt(sos, vx)

        offset = distance_km - X0
        plt.plot(vz_filt + offset, time, color='black')
        #plt.plot(time, vx_filt + offset, color='red')
        
    except Exception as e:
        print(f"Error reading {filename}: {e}")

# 3. グラフの装飾
plt.ylabel('Time [s]')
plt.xlabel('Vz + Offset (Distance) [km]')
plt.grid(True, linestyle='--', alpha=0.5)
# データ数が多い場合は凡例を外に出すか、非表示に
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
plt.xlim(-40,280)
plt.ylim(0,120)
plt.tight_layout()
plt.show()